# 01. Preprocessing — Quora Question Pairs
**Team 5 | Taehun Kim**

이 노트북은 모든 팀원의 **베이스라인 입력 데이터**를 생성합니다.  
각 팀원은 아래 출력 파일을 로드하여 사용하고, 추가 전처리는 각자 노트북에서 진행합니다.

```
outputs/
  splits/        ← 공통: train/val/test 인덱스
  for_ml/        ← Taehoon (02): 수동 피처 CSV + TF-IDF sparse matrix
  for_lstm/      ← Jun (03): 패딩 시퀀스 npy + 단어사전 pkl
  for_bert/      ← Yuchan (04), Sanghun (05): 클리닝된 텍스트 CSV
```

## 0. 환경 설정

In [ ]:
# Kaggle에 없을 수 있는 패키지 설치
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'pyarrow'], check=False)
print('패키지 준비 완료')

In [ ]:
import os, re, pickle, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.sparse as sp

from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

import nltk
nltk.download('stopwords', quiet=True)
warnings.filterwarnings('ignore')

for d in ['outputs/splits', 'outputs/for_ml', 'outputs/for_lstm', 'outputs/for_bert']:
    os.makedirs(d, exist_ok=True)

KAGGLE_PATH = '/kaggle/input/quora-question-pairs/train.csv'
LOCAL_PATH  = 'data/train.csv'
DATA_PATH   = KAGGLE_PATH if os.path.exists(KAGGLE_PATH) else LOCAL_PATH

RANDOM_STATE = 42
MAX_SEQ_LEN  = 50
TFIDF_MAX    = 50000

print(f'Data: {DATA_PATH}')

## 1. 데이터 로딩 & 결측치 처리

In [ ]:
df = pd.read_csv(DATA_PATH)
print(f'Shape: {df.shape}')
print(f'Columns: {list(df.columns)}\n')

print('=== 결측치 ===');
print(df.isnull().sum())

# 결측치 → 빈 문자열
df['question1'] = df['question1'].fillna('')
df['question2'] = df['question2'].fillna('')

print('\n결측치 처리 완료')

## 2. EDA (Exploratory Data Analysis)

In [ ]:
print(f'총 질문 쌍:  {len(df):,}')
print(f'중복(1):    {df.is_duplicate.sum():,}  ({df.is_duplicate.mean():.1%})')
print(f'비중복(0):  {(1-df.is_duplicate).sum():,}  ({1-df.is_duplicate.mean():.1%})')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# 클래스 분포
df['is_duplicate'].value_counts().plot(
    kind='bar', ax=axes[0], color=['steelblue', 'coral'], rot=0
)
axes[0].set_title('Class Distribution')
axes[0].set_xticklabels(['Non-Duplicate', 'Duplicate'])

# 질문 길이 분포 (단어 수)
q1_wc = df['question1'].apply(lambda x: len(x.split()))
q2_wc = df['question2'].apply(lambda x: len(x.split()))
axes[1].hist(q1_wc, bins=50, alpha=0.6, label='Q1', color='steelblue')
axes[1].hist(q2_wc, bins=50, alpha=0.6, label='Q2', color='coral')
axes[1].set_xlim(0, 60)
axes[1].set_title('Word Count Distribution')
axes[1].legend()

# 단어 겹침 비율: 중복 vs 비중복
def _word_share(r):
    q1 = set(str(r['question1']).lower().split())
    q2 = set(str(r['question2']).lower().split())
    return len(q1 & q2) / max(len(q1 | q2), 1)

ws = df.apply(_word_share, axis=1)
ws[df.is_duplicate == 0].hist(bins=40, ax=axes[2], alpha=0.6, label='Non-Dup', color='steelblue')
ws[df.is_duplicate == 1].hist(bins=40, ax=axes[2], alpha=0.6, label='Duplicate', color='coral')
axes[2].set_title('Word Share Ratio')
axes[2].legend()

plt.tight_layout()
plt.savefig('outputs/eda.png', dpi=120)
plt.show()
print('EDA 저장: outputs/eda.png')

## 3. 텍스트 클리닝 함수 정의

In [ ]:
CONTRACTIONS = {
    "what's": "what is",  "what're": "what are", "who's": "who is",
    "where's": "where is", "when's": "when is",  "why's": "why is",
    "how's": "how is",    "it's": "it is",        "he's": "he is",
    "she's": "she is",    "that's": "that is",    "there's": "there is",
    "they're": "they are","we're": "we are",      "you're": "you are",
    "i've": "i have",     "you've": "you have",   "we've": "we have",
    "they've": "they have","i'd": "i would",      "you'd": "you would",
    "he'd": "he would",   "she'd": "she would",   "we'd": "we would",
    "they'd": "they would","i'll": "i will",      "you'll": "you will",
    "he'll": "he will",   "she'll": "she will",   "we'll": "we will",
    "they'll": "they will","i'm": "i am",         "can't": "cannot",
    "couldn't": "could not","don't": "do not",   "doesn't": "does not",
    "didn't": "did not",  "isn't": "is not",     "aren't": "are not",
    "wasn't": "was not",  "weren't": "were not", "won't": "will not",
    "wouldn't": "would not","haven't": "have not","hasn't": "has not",
    "hadn't": "had not",  "should've": "should have",
    "would've": "would have","could've": "could have",
}

def clean_for_ml(text: str) -> str:
    # [anokas] 소문자화 + 축약어 확장 + 구두점 제거 — anokas starter 전처리 기준임
    # [nicapotato] 동일 방식 사용함
    # 숫자는 의미 있을 수 있어서 유지함 (ex. "top 10", "iphone 14")
    if not isinstance(text, str) or text == '':
        return ''
    text = text.lower()
    for k, v in CONTRACTIONS.items():
        text = text.replace(k, v)
    text = re.sub(r'<[^>]+>', ' ', text)
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def clean_for_lstm(text: str) -> str:
    # for_ml과 파이프라인 분리 — Jun이 임베딩에 맞게 독립적으로 수정 가능
    # 현재: GloVe/Word2Vec이 소문자+구두점제거 환경에서 학습됐으므로 동일 방식 적용
    # 향후 Jun이 바꿀 수 있는 것: 숫자 처리 방식, 특수 토큰 추가, stemming 등
    if not isinstance(text, str) or text == '':
        return ''
    text = text.lower()
    for k, v in CONTRACTIONS.items():
        text = text.replace(k, v)
    text = re.sub(r'<[^>]+>', ' ', text)
    text = re.sub(r'[^a-z\s]', ' ', text)  # ML과 차이: 숫자 제거 (임베딩 vocab에 숫자 적음)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def clean_for_bert(text: str) -> str:
    # [vabatista] BERT는 자체 WordPiece tokenizer가 대소문자·구두점 처리하므로 최소 클리닝만 함
    # 소문자화·구두점 제거하면 오히려 BERT 성능 저하될 수 있음
    if not isinstance(text, str) or text == '':
        return ''
    text = re.sub(r'<[^>]+>', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

print('ML/LSTM용 클리닝 중...')
df['q1_clean'] = df['question1'].apply(clean_for_ml)
df['q2_clean'] = df['question2'].apply(clean_for_ml)

print('LSTM용 클리닝 중...')
df['q1_lstm'] = df['question1'].apply(clean_for_lstm)
df['q2_lstm'] = df['question2'].apply(clean_for_lstm)

print('BERT용 클리닝 중...')
df['q1_bert'] = df['question1'].apply(clean_for_bert)
df['q2_bert'] = df['question2'].apply(clean_for_bert)

print('완료')
df[['question1', 'q1_clean', 'q1_lstm', 'q1_bert']].head(3)

## 4. Train / Val / Test Split (공통 — 모든 파트 공유)

In [ ]:
idx = df.index.values
y   = df['is_duplicate'].values

idx_train, idx_temp, _, y_temp = train_test_split(
    idx, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
idx_val, idx_test, _, _ = train_test_split(
    idx_temp, y_temp, test_size=0.5, random_state=RANDOM_STATE, stratify=y_temp
)

# splits/ 에 저장 — git에 커밋되어 팀 전체 동일 split 보장
# 각 팀원은 raw CSV 로드 후 이 인덱스로 슬라이싱 → random_state 일치 여부 증명 불필요
os.makedirs('splits', exist_ok=True)
np.save('splits/train_idx.npy', idx_train)
np.save('splits/val_idx.npy',   idx_val)
np.save('splits/test_idx.npy',  idx_test)

print(f'Train: {len(idx_train):,}  ({len(idx_train)/len(df):.0%})')
print(f'Val:   {len(idx_val):,}  ({len(idx_val)/len(df):.0%})')
print(f'Test:  {len(idx_test):,}  ({len(idx_test)/len(df):.0%})')
print('splits/ 저장 완료 — git push 후 팀원들이 동일 파일 사용')

train_df = df.iloc[idx_train].reset_index(drop=True)
val_df   = df.iloc[idx_val].reset_index(drop=True)
test_df  = df.iloc[idx_test].reset_index(drop=True)

---
## 5. [for_ml] 피처 엔지니어링 + TF-IDF
> Taehoon → `02_traditional_ml.ipynb`

In [ ]:
def extract_features(q1: str, q2: str) -> dict:
    t1 = set(q1.split())
    t2 = set(q2.split())
    common = t1 & t2

    def r(num, denom): return num / denom if denom else 0

    return {
        # [nicapotato] 질문 길이 — 길이 차이가 중복 여부와 상관관계 있음
        'q1_len':       len(q1),
        'q2_len':       len(q2),
        # [anokas] 단어 수 — anokas starter에서 핵심 피처로 사용함
        'q1_n_words':   len(q1.split()),
        'q2_n_words':   len(q2.split()),
        # [nicapotato] 길이 차이 파생 피처
        'abs_len_diff': abs(len(q1) - len(q2)),
        'mean_len':     (len(q1) + len(q2)) / 2,
        # [anokas] 공통 단어 수 — word_share 계산의 중간값
        'word_common':  len(common),
        'word_total':   len(t1 | t2),
        # [anokas] 핵심 피처 — 중복/비중복 구분력 가장 높음
        # 공식: |A∩B| / (|A| + |B|)  ← anokas 원본 기준
        # Jaccard(|A∪B| 분모)와 다름. 동일 단어가 많을수록 두 집합 크기 합이 커져서 패널티 없음
        'word_share':   r(len(common), len(t1) + len(t2)),
    }

def build_features(df_part: pd.DataFrame) -> pd.DataFrame:
    return pd.DataFrame([
        extract_features(r.q1_clean, r.q2_clean)
        for _, r in df_part.iterrows()
    ])

print('피처 함수 정의 완료 (9개 — anokas + nicapotato 베이스라인)')

In [ ]:
print('Train 피처 추출 중...')
X_train_hand = build_features(train_df)
print('Val 피처 추출 중...')
X_val_hand   = build_features(val_df)
print('Test 피처 추출 중...')
X_test_hand  = build_features(test_df)

print(f'피처 shape: {X_train_hand.shape}')
X_train_hand.describe().T[['mean', 'std', 'min', 'max']].head(10)

In [ ]:
# [nicapotato] TF-IDF 기반 텍스트 벡터화 — nicapotato 베이스라인 핵심 방식임
# train만으로 fit — val/test 정보 누수 방지
train_corpus = pd.concat([train_df['q1_clean'], train_df['q2_clean']])

tfidf = TfidfVectorizer(
    max_features=TFIDF_MAX,  # 5K~50K 범위에서 튜닝 예정
    ngram_range=(1, 1),      # [nicapotato] unigram만 사용 — 베이스라인 기준
    stop_words='english',    # [nicapotato] 불용어 제거
    min_df=2                 # 1회만 등장하는 단어 제거 — 노이즈 감소
    # sublinear_tf 미사용 — nicapotato 베이스라인에 없는 enhancement라 제외함
)
tfidf.fit(train_corpus)

def tfidf_hstack(df_part):
    # [nicapotato] q1, q2 각각 변환 후 옆으로 붙임 → 모델이 두 질문을 독립적으로 참조 가능
    q1 = tfidf.transform(df_part['q1_clean'])
    q2 = tfidf.transform(df_part['q2_clean'])
    return sp.hstack([q1, q2], format='csr')

def cosine_col(df_part):
    # [nicapotato] TF-IDF 벡터 간 코사인 유사도 — 수동 피처로 추가함
    q1 = tfidf.transform(df_part['q1_clean'])
    q2 = tfidf.transform(df_part['q2_clean'])
    return np.array([cosine_similarity(q1[i], q2[i])[0][0] for i in range(len(df_part))])

print('TF-IDF 변환 중...')
X_train_tfidf = tfidf_hstack(train_df)
X_val_tfidf   = tfidf_hstack(val_df)
X_test_tfidf  = tfidf_hstack(test_df)

print('코사인 유사도 피처 생성 중...')
X_train_hand['tfidf_cosine'] = cosine_col(train_df)
X_val_hand['tfidf_cosine']   = cosine_col(val_df)
X_test_hand['tfidf_cosine']  = cosine_col(test_df)

print(f'TF-IDF sparse shape: {X_train_tfidf.shape}')
print('최종 수동 피처: 9개 + tfidf_cosine = 10개')

In [ ]:
# 레이블 저장
np.save('outputs/for_ml/y_train.npy', train_df['is_duplicate'].values)
np.save('outputs/for_ml/y_val.npy',   val_df['is_duplicate'].values)
np.save('outputs/for_ml/y_test.npy',  test_df['is_duplicate'].values)

# 수동 피처 저장
X_train_hand.to_parquet('outputs/for_ml/X_train_hand.parquet')
X_val_hand.to_parquet('outputs/for_ml/X_val_hand.parquet')
X_test_hand.to_parquet('outputs/for_ml/X_test_hand.parquet')

# TF-IDF sparse matrix 저장
sp.save_npz('outputs/for_ml/X_train_tfidf.npz', X_train_tfidf)
sp.save_npz('outputs/for_ml/X_val_tfidf.npz',   X_val_tfidf)
sp.save_npz('outputs/for_ml/X_test_tfidf.npz',  X_test_tfidf)

# TF-IDF vectorizer 저장 (재사용 가능)
with open('outputs/for_ml/tfidf_vectorizer.pkl', 'wb') as f:
    pickle.dump(tfidf, f)

print('[for_ml] 저장 완료')
print(f'  수동 피처: {X_train_hand.shape[1]}개')
print(f'  TF-IDF: {X_train_tfidf.shape}')

---
## 6. [for_lstm] 시퀀스 전처리
> Jun Park → `03_siamese_lstm.ipynb`

> ⚠️ **베이스라인 근거 없음** — 세 지정 베이스라인(anokas/nicapotato/vabatista) 중 LSTM 방식 없음.  
> Word2Vec/GloVe + Siamese LSTM은 일반 NLP 관행 기준으로 구현함.  
> 텍스트 클리닝은 `clean_for_ml`(anokas 기준)을 재사용함.


In [ ]:
# train 데이터로만 단어 사전 구축 (data leakage 방지)
# q1_lstm/q2_lstm 사용 — ML 파이프라인(q1_clean)과 독립된 vocab
train_tokens = (
    train_df['q1_lstm'].str.split().explode().tolist() +
    train_df['q2_lstm'].str.split().explode().tolist()
)
word_freq = Counter(train_tokens)

# 빈도 2 미만 단어는 UNK 처리
vocab    = ['<PAD>', '<UNK>'] + [w for w, c in word_freq.most_common() if c >= 2]
word2idx = {w: i for i, w in enumerate(vocab)}
idx2word = {i: w for w, i in word2idx.items()}

print(f'Vocab size: {len(word2idx):,}  (빈도 2 이상)')

with open('outputs/for_lstm/word2idx.pkl', 'wb') as f:
    pickle.dump(word2idx, f)
with open('outputs/for_lstm/idx2word.pkl', 'wb') as f:
    pickle.dump(idx2word, f)

In [ ]:
def encode_pad(texts: pd.Series, word2idx: dict, max_len: int) -> np.ndarray:
    unk = word2idx['<UNK>']
    pad = word2idx['<PAD>']
    result = []
    for text in texts:
        tokens = text.split()[:max_len]
        ids = [word2idx.get(t, unk) for t in tokens]
        ids += [pad] * (max_len - len(ids))
        result.append(ids)
    return np.array(result, dtype=np.int32)

for name, part in [('train', train_df), ('val', val_df), ('test', test_df)]:
    q1 = encode_pad(part['q1_lstm'], word2idx, MAX_SEQ_LEN)  # q1_lstm 사용
    q2 = encode_pad(part['q2_lstm'], word2idx, MAX_SEQ_LEN)  # q2_lstm 사용
    y  = part['is_duplicate'].values.astype(np.int32)
    np.save(f'outputs/for_lstm/{name}_q1.npy', q1)
    np.save(f'outputs/for_lstm/{name}_q2.npy', q2)
    np.save(f'outputs/for_lstm/{name}_y.npy',  y)
    print(f'[{name}] q1/q2: {q1.shape}')

print('[for_lstm] 저장 완료')

---
## 7. [for_bert] 최소 전처리 CSV

In [ ]:
for name, part in [('train', train_df), ('val', val_df), ('test', test_df)]:
    out = part[['q1_bert', 'q2_bert', 'is_duplicate']].copy()
    out.columns = ['question1', 'question2', 'label']
    out.to_csv(f'outputs/for_bert/{name}.csv', index=False)
    print(f'[{name}] {len(out):,} rows  →  outputs/for_bert/{name}.csv')

print('[for_bert] 저장 완료')

## 8. 출력 파일 요약

In [ ]:
print('=== 생성된 파일 목록 ===')
for root, dirs, files in os.walk('outputs'):
    for fname in sorted(files):
        path = os.path.join(root, fname)
        size = os.path.getsize(path) / 1024
        print(f'  {path:<58}  {size:>8.1f} KB')

print()
print('=== 다른 노트북에서 로드하는 방법 ===')
print("""
[02_traditional_ml.ipynb — Taehoon]
  X_train = pd.read_parquet('outputs/for_ml/X_train_hand.parquet')
  X_train_tfidf = sp.load_npz('outputs/for_ml/X_train_tfidf.npz')
  y_train = np.load('outputs/for_ml/y_train.npy')

[03_siamese_lstm.ipynb — Jun]
  train_q1 = np.load('outputs/for_lstm/train_q1.npy')
  train_q2 = np.load('outputs/for_lstm/train_q2.npy')
  with open('outputs/for_lstm/word2idx.pkl','rb') as f: word2idx = pickle.load(f)

[04_bert_crossencoder.ipynb — Yuchan]
[05_biencoder_quantization.ipynb — Sanghun]
  train_df = pd.read_csv('outputs/for_bert/train.csv')
  # → BertTokenizer 적용 후 모델 학습
""")